In [ ]:
#@title 圧縮ツール

import copy
import io
import itertools
import json
import os
import re
import string
import sys
import zipfile
from collections import Counter

SHORTEN_IDS = True
ZERO_BLOCK_XY = True

ID_DICTS = ('blocks', 'comments', 'variables', 'lists', 'broadcasts')
MAX_ERRORS = 10


def _is_num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)


_INDEX_RE = re.compile(r'0|[1-9][0-9]{0,9}')


def is_index_like(s):
    return bool(_INDEX_RE.fullmatch(s)) and int(s) < 4294967295


_HEAD = string.ascii_letters
_TAIL = string.ascii_letters + string.digits


def short_id(n):
    length, size = 1, len(_HEAD)
    while n >= size:
        n -= size
        length += 1
        size *= len(_TAIL)
    chars = []
    for _ in range(length - 1):
        n, r = divmod(n, len(_TAIL))
        chars.append(_TAIL[r])
    chars.append(_HEAD[n])
    return ''.join(reversed(chars))


def _remap_ref(v, f):
    if isinstance(v, str):
        return f(v)
    if isinstance(v, list) and len(v) >= 3 and v[0] in (11, 12, 13) and isinstance(v[2], str):
        return v[:2] + [f(v[2])] + v[3:]
    return v


def _remap_input(inp, f):
    if isinstance(inp, list) and inp:
        return [inp[0]] + [_remap_ref(x, f) for x in inp[1:]]
    return inp


def _remap_field(fld, f):
    if isinstance(fld, list) and len(fld) >= 2 and isinstance(fld[1], str):
        return [fld[0], f(fld[1])] + fld[2:]
    return fld


def _remap_block(b, f, zero_xy):
    if isinstance(b, list):
        b = list(b)
        if len(b) >= 3 and b[0] in (11, 12, 13) and isinstance(b[2], str):
            b[2] = f(b[2])
        if zero_xy:
            for i in (3, 4):
                if i < len(b) and _is_num(b[i]):
                    b[i] = 0
        return b
    if not isinstance(b, dict):
        return b
    b = dict(b)
    for k in ('next', 'parent', 'comment'):
        if isinstance(b.get(k), str):
            b[k] = f(b[k])
    if isinstance(b.get('inputs'), dict):
        b['inputs'] = {k: _remap_input(v, f) for k, v in b['inputs'].items()}
    if isinstance(b.get('fields'), dict):
        b['fields'] = {k: _remap_field(v, f) for k, v in b['fields'].items()}
    if zero_xy:
        for k in ('x', 'y'):
            if _is_num(b.get(k)):
                b[k] = 0
    return b


def _remap_comment(c, f):
    if isinstance(c, dict) and isinstance(c.get('blockId'), str):
        c = dict(c)
        c['blockId'] = f(c['blockId'])
    return c


def _remap_target(t, f, zero_xy):
    if not isinstance(t, dict):
        return t
    t = dict(t)
    if isinstance(t.get('blocks'), dict):
        t['blocks'] = {f(k): _remap_block(b, f, zero_xy) for k, b in t['blocks'].items()}
    if isinstance(t.get('comments'), dict):
        t['comments'] = {f(k): _remap_comment(c, f) for k, c in t['comments'].items()}
    for key in ('variables', 'lists', 'broadcasts'):
        if isinstance(t.get(key), dict):
            t[key] = {f(k): v for k, v in t[key].items()}
    return t


def _remap_monitor(m, f):
    if isinstance(m, dict) and isinstance(m.get('id'), str):
        m = dict(m)
        m['id'] = f(m['id'])
    return m


def remap_project(project, f, zero_xy=False):
    new = dict(project)
    new['targets'] = [_remap_target(t, f, zero_xy) for t in project['targets']]
    if isinstance(project.get('monitors'), list):
        new['monitors'] = [_remap_monitor(m, f) for m in project['monitors']]
    return new


def count_strings(root):
    c = Counter()
    stack = [root]
    while stack:
        n = stack.pop()
        if isinstance(n, str):
            c[n] += 1
        elif isinstance(n, list):
            stack.extend(n)
        elif isinstance(n, dict):
            c.update(n.keys())
            stack.extend(n.values())
    return c


def build_idmap(project):
    defined = {}
    for t in project['targets']:
        if isinstance(t, dict):
            for key in ID_DICTS:
                if isinstance(t.get(key), dict):
                    defined.update(dict.fromkeys(t[key]))

    id_occ = Counter()

    def count(s):
        id_occ[s] += 1
        return s

    remap_project(project, count)
    all_occ = count_strings(project)

    cands = [s for s in defined if id_occ[s] == all_occ[s] and not is_index_like(s)]
    cands.sort(key=lambda s: -id_occ[s])

    fresh = (s for s in map(short_id, itertools.count()) if s not in all_occ)
    nxt = next(fresh)
    idmap = {}
    for old in cands:
        if len(nxt) < len(old):
            idmap[old] = nxt
            nxt = next(fresh)
    return idmap


def compress(project, shorten_ids=True, zero_xy=True):
    idmap = build_idmap(project) if shorten_ids else {}
    return remap_project(project, lambda s: idmap.get(s, s), zero_xy), idmap


def dumps(project):
    kw = dict(separators=(',', ':'), allow_nan=False)
    try:
        return json.dumps(project, ensure_ascii=False, **kw).encode('utf-8')
    except UnicodeEncodeError:
        return json.dumps(project, ensure_ascii=True, **kw).encode('ascii')


def verify(orig, comp, zero_xy):
    errors = []
    fwd, bwd = {}, {}
    defined = set()
    for t in orig['targets']:
        if isinstance(t, dict):
            for key in ID_DICTS:
                if isinstance(t.get(key), dict):
                    defined.update(t[key])

    def err(msg):
        if len(errors) < MAX_ERRORS:
            errors.append(msg)

    def where(path):
        return '/'.join(map(str, path)) or '(root)'

    def link(a, b, path):
        if a != b and a not in defined:
            err(f'{where(path)}: ID以外の文字列が変わっています {a!r} -> {b!r}')
        elif fwd.setdefault(a, b) != b:
            err(f'{where(path)}: 同じID {a!r} が {fwd[a]!r} と {b!r} の2通りに変わっています')
        elif bwd.setdefault(b, a) != a:
            err(f'{where(path)}: 別々のID {bwd[b]!r} と {a!r} が同じ {b!r} になっています')

    def walk(a, b, path):
        if len(errors) >= MAX_ERRORS:
            return
        if (zero_xy and len(path) == 5 and path[0] == 'targets' and path[2] == 'blocks'
                and path[4] in ('x', 'y', 3, 4) and _is_num(a) and _is_num(b) and b == 0):
            return
        if type(a) is not type(b):
            err(f'{where(path)}: 型が変わっています')
        elif isinstance(a, dict):
            if len(a) != len(b):
                err(f'{where(path)}: 要素数が変わっています ({len(a)} -> {len(b)})')
                return
            id_keyed = len(path) == 3 and path[0] == 'targets' and path[2] in ID_DICTS
            for (ka, va), (kb, vb) in zip(a.items(), b.items()):
                if id_keyed:
                    link(ka, kb, path)
                elif ka != kb:
                    err(f'{where(path)}: キー名が変わっています {ka!r} -> {kb!r}')
                    continue
                walk(va, vb, path + (ka,))
        elif isinstance(a, list):
            if len(a) != len(b):
                err(f'{where(path)}: 配列の長さが変わっています ({len(a)} -> {len(b)})')
                return
            for i, (x, y) in enumerate(zip(a, b)):
                walk(x, y, path + (i,))
        elif isinstance(a, str):
            link(a, b, path)
        elif a != b:
            err(f'{where(path)}: 値が変わっています {a!r} -> {b!r}')

    walk(orig, comp, ())
    return errors


def build_sb3(zin, project_json):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w') as zout:
        for item in zin.infolist():
            is_proj = item.filename == 'project.json'
            data = project_json if is_proj else zin.read(item.filename)
            info = copy.copy(item)
            ctype = zipfile.ZIP_DEFLATED if is_proj else item.compress_type
            zout.writestr(info, data, compress_type=ctype, compresslevel=9)
    return buf.getvalue()


def check_sb3(sb3_bytes, zin, project_json):
    problems = []
    try:
        z = zipfile.ZipFile(io.BytesIO(sb3_bytes))
    except zipfile.BadZipFile as e:
        return [f'出力したzipを開けません: {e}']
    with z:
        bad = z.testzip()
        if bad:
            problems.append(f'zip内のファイルが壊れています: {bad}')
        if len(z.namelist()) != len(zin.namelist()):
            problems.append('zip内のファイル数が元と違います')
        for item in zin.infolist():
            try:
                info = z.getinfo(item.filename)
            except KeyError:
                problems.append(f'ファイルが欠けています: {item.filename}')
                continue
            if item.filename == 'project.json':
                if z.read('project.json') != project_json:
                    problems.append('project.json が検証済みの内容と一致しません')
            elif info.CRC != item.CRC or info.file_size != item.file_size:
                problems.append(f'ファイルの内容が元と違います: {item.filename}')
    return problems


def attempt_levels(shorten_ids, zero_block_xy):
    levels = [(shorten_ids, zero_block_xy)]
    if shorten_ids:
        levels.append((False, zero_block_xy))
    if zero_block_xy:
        levels.append((False, False))
    return list(dict.fromkeys(levels))


def process(name, data, shorten_ids=None, zero_block_xy=None):
    shorten_ids = SHORTEN_IDS if shorten_ids is None else shorten_ids
    zero_block_xy = ZERO_BLOCK_XY if zero_block_xy is None else zero_block_xy
    base, ext = os.path.splitext(os.path.basename(name))
    zin = None
    if ext.lower() == '.json':
        raw = data
    else:
        try:
            zin = zipfile.ZipFile(io.BytesIO(data))
            raw = zin.read('project.json')
        except (zipfile.BadZipFile, KeyError) as e:
            raise ValueError(f'sb3として読めません（zipが壊れているか project.json がありません）: {e!r}')

    orig = json.loads(raw.decode('utf-8-sig'))
    if not (isinstance(orig, dict) and isinstance(orig.get('targets'), list)):
        raise ValueError('Scratch 3.0 形式の project.json ではありません（targets がありません）')

    for shorten, zero_xy in attempt_levels(shorten_ids, zero_block_xy):
        comp, idmap = compress(orig, shorten, zero_xy)
        out = dumps(comp)
        problems = verify(orig, json.loads(out.decode('utf-8')), zero_xy)
        if not problems:
            break
        print(f'[NG] ID短縮={shorten} 座標0化={zero_xy} → 再試行')
        for p in problems:
            print('  ' + p)
    else:
        raise RuntimeError('検証に通らないため出力しません')

    if len(out) >= len(raw):
        out = raw
    print(f'project.json: {len(raw):,} → {len(out):,} bytes ({(1 - len(out) / max(len(raw), 1)) * 100:.1f}%減)  検証OK')

    if zin is None:
        return f'{base}_min.json', out

    new_zip = build_sb3(zin, out)
    problems = check_sb3(new_zip, zin, out)
    if problems:
        raise RuntimeError('sb3チェック失敗: ' + ' / '.join(problems[:MAX_ERRORS]))
    print(f'sb3: {len(data):,} → {len(new_zip):,} bytes  zip検証OK')
    return f'{base}_min.sb3', new_zip


def main(paths=None, shorten_ids=None, zero_block_xy=None):
    try:
        from google.colab import files
        in_colab = True
    except ImportError:
        in_colab = False

    if in_colab:
        uploaded = files.upload()
    else:
        uploaded = {}
        for p in paths or []:
            with open(p, 'rb') as fh:
                uploaded[p] = fh.read()
    if not uploaded:
        print('ファイルが選択されていません')
        return

    for name, data in uploaded.items():
        print(f'\n{name}')
        try:
            out_name, out_bytes = process(name, data, shorten_ids, zero_block_xy)
        except Exception as e:
            print(f'[出力なし] {type(e).__name__}: {e}')
            continue
        with open(out_name, 'wb') as fh:
            fh.write(out_bytes)
        print(f'→ {out_name}')
        if in_colab:
            files.download(out_name)


In [ ]:
#@title 実行
SHORTEN_IDS = True    #@param {type:"boolean"}
ZERO_BLOCK_XY = True  #@param {type:"boolean"}

main(shorten_ids=SHORTEN_IDS, zero_block_xy=ZERO_BLOCK_XY)
